# Memoria del Proyecto: Resolución Automática de Sudokus a partir de Imágenes


## Objetivo

El objetivo del proyecto ha sido construir un sistema capaz de recibir una imagen de un Sudoku, detectar el tablero, leer los números presentes en las celdas, construir una matriz 9x9 y generar una propuesta de solución.

El proyecto se ha dividido en varias fases independientes para facilitar el desarrollo, la validación y la integración final en una aplicación Streamlit.

## Arquitectura general del sistema

La solución final se ha organizado como un pipeline compuesto por varios módulos:

```text
Imagen de entrada
↓
Modelo YOLO para detectar el tablero
↓
Recorte del Sudoku
↓
Segmentación en 81 celdas
↓
Modelo CNN para reconocer números
↓
Matriz 9x9 inicial
↓
Modelo predictivo de resolución
↓
Matriz solución predicha
↓
Aplicación Streamlit
```

Esta división permitió trabajar cada problema por separado: detección de objetos, segmentación de imagen, clasificación de dígitos y resolución del Sudoku.

## Fase 1. Detección del tablero mediante YOLO

En el primer notebook, `01_entrenamiento_yolo.ipynb`, se entrenó un modelo YOLO para localizar automáticamente el tablero del Sudoku dentro de una imagen completa.

### Decisiones tomadas

- Se utilizó YOLO porque la primera tarea era un problema claro de detección de objetos: localizar el área del tablero dentro de una imagen.
- El modelo se entrenó para detectar una única clase: `sudoku`.
- Una vez entrenado, el modelo generó un archivo `best.pt`, que se guardó en la carpeta `models/`.
- Para continuar el pipeline no se utilizó la imagen dibujada por YOLO con la caja de detección, sino las coordenadas reales de la bounding box para recortar el tablero desde la imagen original.

### Resultado de esta fase

El resultado principal de esta fase fue un recorte limpio del Sudoku, sin etiquetas, bordes azules ni elementos externos de la página web. Este recorte se utilizó como entrada de la fase de segmentación.

## Fase 2. Segmentación del tablero y reconocimiento de dígitos

El segundo notebook, `02_lectura_celdas_sudoku.ipynb`, se centró en convertir el recorte del tablero en una matriz 9x9. Esta fase fue una de las más importantes del proyecto porque conecta la parte visual con la parte lógica o predictiva.

El objetivo era pasar de una imagen recortada del Sudoku a una estructura como esta:

```python
[
 [0, 0, 6, 0, 0, 0, 8, 0, 2],
 [7, 0, 0, 4, 2, 8, 0, 9, 6],
 ...
]
```

Donde `0` representa una celda vacía y los valores `1-9` representan los números detectados.

### 2.1 Segmentación en 81 celdas

Una vez obtenido el recorte limpio del Sudoku, se decidió segmentar el tablero de forma geométrica, sin entrenar un segundo modelo YOLO para detectar cada número.

La decisión se basó en que el modelo YOLO ya recortaba bastante bien los límites del tablero y el Sudoku aparecía prácticamente frontal y alineado.

El proceso fue:

1. Redimensionar el recorte del Sudoku a `450x450` píxeles.
2. Dividir la imagen en una cuadrícula de `9x9`.
3. Obtener celdas de `50x50` píxeles.
4. Aplicar un margen interior para reducir la presencia de líneas de la cuadrícula.

Se probaron distintos márgenes:

- `margin = 4`: conservaba bien los números, pero dejaba más líneas de la cuadrícula.
- `margin = 8`: eliminaba más líneas, pero recortaba partes de algunos números.
- `margin = 5`: fue el compromiso elegido, porque mantenía los números completos y reducía parte de las líneas.

La salida de esta fase fueron 81 imágenes individuales, guardadas con nombres del tipo:

```text
cell_0_0.png
cell_0_1.png
...
cell_8_8.png
```

Esta segmentación se consideró suficientemente buena para continuar.

### 2.2 Preprocesado de las celdas

Antes de intentar clasificar los números, se aplicó un preprocesado para transformar cada celda en una imagen más adecuada para una red neuronal.

Los pasos fueron:

1. Conversión a escala de grises.
2. Suavizado con `GaussianBlur`.
3. Binarización con `adaptiveThreshold`.
4. Inversión de colores para obtener número blanco sobre fondo negro.
5. Recorte del centro de la celda para eliminar restos de líneas de la cuadrícula.
6. Redimensionado final a `28x28` píxeles.
7. Normalización de valores entre `0` y `1`.

Durante esta fase se probaron distintos recortes centrales:

- Recortar del 15% al 85% eliminaba muchas líneas, pero algunos números quedaban demasiado ajustados o parcialmente cortados.
- Recortar del 10% al 90% conservaba mejor la forma completa de los dígitos y mantenía bajo el ruido de los bordes.

Se eligió finalmente el recorte central `10%-90%`.

## 2.3 Alternativas probadas para la detección de números

Para convertir cada imagen de celda en un número, se probaron tres enfoques principales. Esta parte fue especialmente relevante porque permitió comprobar que reconocer los números del Sudoku no era tan directo como aplicar un modelo genérico de dígitos.

Las tres alternativas fueron:

1. Modelo entrenado con MNIST.
2. Modelo entrenado con dígitos sintéticos generados por código.
3. Modelo entrenado con celdas reales del propio Sudoku.

### Alternativa 1. Modelo CNN entrenado con MNIST

La primera aproximación fue utilizar el dataset MNIST, que contiene imágenes de números manuscritos del 0 al 9. Se entrenó una CNN sencilla con la siguiente estructura:

```text
Conv2D
↓
MaxPooling2D
↓
Conv2D
↓
MaxPooling2D
↓
Flatten
↓
Dense
↓
Softmax(10 clases)
```

La idea era aprovechar un dataset clásico de reconocimiento de dígitos y aplicar el modelo a las celdas del Sudoku.

### Problema encontrado

Aunque conceptualmente parecía una opción razonable, en la práctica no funcionó bien. El motivo principal fue la diferencia entre los datos de entrenamiento y los datos reales del proyecto:

- MNIST contiene números manuscritos.
- Los números del Sudoku proceden de una fuente digital fija.
- El grosor, la forma y el centrado de los dígitos son muy distintos.
- En Sudoku, el valor `0` no significa el número cero, sino una celda vacía.

Por tanto, el modelo aprendía a reconocer números manuscritos, pero no generalizaba correctamente a los números impresos del Sudoku. La matriz resultante contenía muchos errores.

### Alternativa 2. Dataset sintético generado con OpenCV

Tras comprobar que MNIST no generalizaba bien, se intentó crear un dataset artificial más parecido al problema real. Para ello se generaron imágenes de `28x28` píxeles usando OpenCV.

El proceso fue:

- Crear imágenes negras de fondo.
- Dibujar números blancos con `cv2.putText`.
- Variar ligeramente escala, grosor, posición y ruido.
- Generar también ejemplos de clase `0`, entendida como celda vacía.
- Crear un conjunto equilibrado con el mismo número de ejemplos por clase.

La ventaja de este enfoque era que no requería etiquetado manual y permitía generar miles de ejemplos rápidamente.

### Problema encontrado

A pesar de ser más parecido que MNIST, el dataset sintético tampoco funcionó suficientemente bien. Los números generados con OpenCV no coincidían exactamente con la tipografía, posición y forma de los números reales del Sudoku.

Además, el modelo aprendía sobre imágenes demasiado artificiales, mientras que las celdas reales conservaban pequeños restos de líneas, variaciones de fondo y formas concretas de la web original.

Como resultado, la matriz predicha seguía siendo poco fiable e incluso en algunos casos empeoró respecto al intento con MNIST.

### Alternativa 3. Entrenamiento con celdas reales del Sudoku

La tercera alternativa consistió en entrenar el modelo con las propias celdas reales extraídas del Sudoku. Para ello se utilizó el Sudoku que ya estaba segmentado y se definió manualmente la cadena `puzzle_str` con los 81 valores reales.

Esta cadena permitía asociar cada celda `cell_fila_columna.png` con su etiqueta correcta:

```text
0 = celda vacía
1-9 = número presente en la celda
```

Después se creó un pequeño dataset por clase a partir de las celdas reales y se aplicó `data augmentation` para generar variaciones:

- Pequeñas rotaciones.
- Pequeños desplazamientos horizontales y verticales.
- Conservación del fondo negro.
- Conservación del estilo real de los números.

Este enfoque tenía una ventaja importante: el modelo ya no aprendía de números manuscritos ni de números artificiales, sino de imágenes reales del propio proyecto.

### Resultado de la alternativa 3

Esta alternativa fue la más coherente con el problema, pero también presentó una limitación clara: solo se disponía de un Sudoku etiquetado, es decir, únicamente 81 celdas reales originales.

Aunque se generaron variaciones mediante `data augmentation`, el número de ejemplos reales de partida era demasiado bajo para conseguir un clasificador robusto.

El modelo final de dígitos se guardó como:

```text
models/modelo_digitos_sudoku.keras
```

Sin embargo, se asumió que su precisión era limitada. Aun así, se decidió continuar con este modelo porque el objetivo principal era completar el flujo completo del proyecto hasta Streamlit.

### Conclusión sobre la lectura de números

La fase de reconocimiento de dígitos permitió comprobar tres ideas importantes:

1. Un modelo genérico como MNIST no siempre es válido si el dominio de aplicación cambia.
2. Los datos sintéticos ayudan, pero deben parecerse mucho al dominio real.
3. Para que la CNN de dígitos funcione bien sería necesario disponer de muchas más celdas reales etiquetadas.

La mejora futura más clara sería generar un dataset real con muchas imágenes de Sudokus, segmentarlas automáticamente y etiquetar las celdas para entrenar un clasificador más robusto.

## Fase 3. Resolución del Sudoku mediante Deep Learning

En el tercer notebook, `03_resolver_sudoku.ipynb`, se trabajó con un dataset de aproximadamente 9 millones de Sudokus. Cada registro contenía dos columnas:

- `puzzle`: Sudoku incompleto codificado como cadena de 81 caracteres.
- `solution`: Sudoku completo codificado como cadena de 81 caracteres.

Un punto importante fue cargar estas columnas como texto (`str`), ya que Pandas las interpretaba inicialmente como enteros y se podían perder ceros iniciales.

### Preparación de datos

Para reducir tiempos de entrenamiento se trabajó con una muestra de datos en lugar de utilizar los 9 millones de registros completos.

Se probaron dos tamaños principales:

- 100.000 registros.
- 300.000 registros.

Cada cadena de 81 caracteres se transformó en un array numérico de tamaño 81. La entrada se normalizó dividiendo por 9, mientras que la salida se ajustó restando 1 a las etiquetas porque el modelo tenía 9 clases por celda, numeradas internamente de 0 a 8.

### Modelo predictivo de resolución

Se construyó una red neuronal densa que recibía 81 valores de entrada y devolvía 81 predicciones, una por cada celda del Sudoku resuelto.

La salida del modelo tenía forma:

```text
(81, 9)
```

Es decir, para cada una de las 81 posiciones, el modelo estimaba la probabilidad de que el valor correcto fuera uno de los números del 1 al 9.

El modelo consiguió mejorar al aumentar los datos y el número de épocas, llegando aproximadamente a un 45% de precisión de validación con 300.000 registros y 25 épocas.

### Limitación del modelo predictivo

Aunque el modelo aprendió patrones del Sudoku, no garantizaba que la solución final cumpliera las reglas del juego. En algunas predicciones aparecían números repetidos en filas, columnas o subcuadrículas.

Esto se debe a que el Sudoku es un problema determinista basado en restricciones. Una red neuronal predictiva puede aproximar patrones, pero no asegura por sí misma que se respeten todas las reglas.

Aun así, se decidió guardar este modelo como parte del proyecto porque cumplía el requisito de entrenar un modelo con los Sudokus resueltos disponibles.

El modelo se guardó como:

```text
models/modelo_solver_sudoku_v2.keras
```

## Desarrollo de la aplicación Streamlit

Como entrega final se desarrolló una aplicación sencilla en Streamlit. La app permite subir una imagen de un Sudoku y ejecutar el pipeline completo.

El flujo implementado fue:

```text
Subida de imagen
↓
Carga de modelos
↓
Detección YOLO del tablero
↓
Recorte del Sudoku
↓
Segmentación en 81 celdas
↓
Reconocimiento de dígitos mediante CNN
↓
Construcción de matriz 9x9
↓
Predicción de solución mediante modelo solver
↓
Visualización de resultados
```

Los modelos utilizados por la aplicación fueron:

```text
models/best.pt
models/modelo_digitos_sudoku.keras
models/modelo_solver_sudoku_v2.keras
```

## Estructura final del proyecto

La estructura final del proyecto se organizó separando notebooks, modelos y aplicación:

```text
sudoku_project/
│
├── notebooks/
│   ├── 01_entrenamiento_yolo.ipynb
│   ├── 02_lectura_celdas_sudoku.ipynb
│   └── 03_resolver_sudoku.ipynb
│
├── models/
│   ├── best.pt
│   ├── modelo_digitos_sudoku.keras
│   └── modelo_solver_sudoku_v2.keras
│
├── app.py
└── README.md
```

Los modelos se guardaron en una carpeta independiente porque, una vez entrenados, no dependen directamente de los datos originales para hacer predicciones. Los datos solo serían necesarios si se quisiera reentrenar o mejorar los modelos.

## Limitaciones del proyecto

Las principales limitaciones detectadas fueron:

1. **Reconocimiento de números limitado**: el modelo de dígitos se entrenó con muy pocas celdas reales etiquetadas, por lo que su precisión no fue alta.
2. **Modelo solver no determinista**: el modelo predictivo de resolución no garantiza que las soluciones cumplan las reglas del Sudoku.
3. **Dependencia de la calidad del recorte**: aunque YOLO funcionó bien, errores en el recorte afectarían a la segmentación de las celdas.
4. **Dataset visual insuficiente**: se disponía de muchas imágenes de Sudokus, pero no de sus etiquetas asociadas celda a celda, por lo que no se pudo crear automáticamente un dataset grande para el reconocimiento de números.

## Posibles mejoras futuras

Las principales mejoras futuras serían:

- Crear un dataset real de celdas etiquetadas a partir de muchas imágenes de Sudokus.
- Reentrenar la CNN de dígitos con más ejemplos reales.
- Añadir validaciones para detectar matrices iniciales inconsistentes.
- Combinar el modelo predictivo con un solver lógico basado en reglas para garantizar soluciones válidas.
- Mejorar la interfaz Streamlit mostrando también las 81 celdas segmentadas y permitiendo corregir manualmente errores de lectura antes de resolver.
- Entrenar el solver con arquitecturas más adecuadas para restricciones, como modelos basados en grafos o enfoques híbridos.

## Conclusiones

El proyecto permitió construir un flujo completo desde una imagen de Sudoku hasta una solución predicha. Se entrenaron tres tipos de modelos o componentes principales:

1. Un detector YOLO para localizar el tablero.
2. Una CNN para reconocer dígitos en las celdas.
3. Un modelo predictivo para generar una solución a partir de la matriz inicial.

Aunque los resultados de reconocimiento de números y resolución no fueron perfectos, se consiguió implementar un sistema funcional de extremo a extremo y se identificaron claramente las limitaciones técnicas del enfoque.

La principal conclusión es que la detección del tablero funcionó correctamente, mientras que las fases de lectura de dígitos y resolución requieren más datos o un enfoque híbrido para alcanzar resultados robustos.